In [1]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import deeplake 
import torch



class patches_loader(Dataset):
    '''
    Class to handle patches stored as deeplake objects
    '''
    def __init__(self, train_or_test, WSI_id, scanner, to_torch=False, emb=True):
        self.train_or_test = train_or_test
        self.WSI_id = WSI_id
        self.scanner = scanner
        directory = f"/home/leolr-int/nfs/data/data/patched/dim_256/{train_or_test}"
        #here we consider only Subset3
        WSI = f'Subset3_{train_or_test}_{WSI_id}_{scanner}'
        self.patches = deeplake.open_read_only(f'{directory}/{WSI}')
        self.to_torch = to_torch
        self.emb = emb

    def summary(self):
        return self.patches.summary()
    
    # to specify a label and then access the columns of the deeplake dataset
    def __getitem__(self, idx):
        if self.to_torch:
            patch = self.patches[idx]
            img = torch.tensor(patch['patch'])
            label = torch.tensor(patch["label"], dtype=torch.long)
            area = torch.tensor(patch["area"], dtype=torch.long)
            x = torch.tensor(patch["x"], dtype=torch.long)
            y = torch.tensor(patch["y"], dtype=torch.long)
            w = torch.tensor(patch["w"], dtype=torch.long)
            h = torch.tensor(patch["h"], dtype=torch.long)
            
            metadata = {
                "area": area,
                "x": x,
                "y": y,
                "w": w,
                "h": h,
                }
            
            dic = {'img':img, 'label':label, 'metadata': metadata}

            if self.emb:
                # connection to embeddings
                directory = '/home/leolr-int/nfs/transformed_data/my_embeddings'
                WSI = f'Subset3_{self.train_or_test}_{self.WSI_id}_{self.scanner}'
                embedding_ds = deeplake.open_read_only(f'{directory}/{WSI}')
                embedding = embedding_ds[idx]['embedding']
                embedding = torch.tensor(embedding, dtype=torch.float)
                dic['embedding'] = embedding
            
            return dic
            
        else: 
            #deeplake object
            return self.patches[idx]
            # Example: patches[idx]['label']

    def __len__(self):
        return len(self.patches)
    
    def display(self, idx): 
        fig, axes = plt.subplots(figsize=(4, 4))
        axes.imshow(self.patches[idx]["patch"])
        plt.show()

    
    def to_embedding(self, idx=None):
        # connection to embeddings
        directory = '/home/leolr-int/nfs/transformed_data/my_embeddings'
        WSI = f'Subset3_{self.train_or_test}_{self.WSI_id}_{self.scanner}'
        embedding_ds = deeplake.open_read_only(f'{directory}/{WSI}')
        if idx == None:
            embeddings_np = np.array(embedding_ds['embedding'])  # stack into 1 array
            embeddings_tensor = torch.from_numpy(embeddings_np).float()
            return embeddings_tensor 
        else:
            embedding = embedding_ds[idx]['embedding']
            embedding = torch.tensor(embedding, dtype=torch.long)
        return embedding
      




In [2]:
#test

idx=1500
file_test = patches_loader('Train', 1, 'Akoya', to_torch=True)
#file_test.display(idx)
#file_test.summary()
#print(file_test.to_embedding())
file_test[idx]['embedding']


tensor([ 0.3235, -0.3488, -0.2578,  ...,  0.5007,  1.7813, -0.2065])

In [3]:
class multi_WSI_loader(Dataset):
    '''
    Class to handle several WSI from different scanners
    Used for training a neural network
    '''

    def __init__(self, WSI_ids, scanners, train_or_test='Train'):
        self.train_or_test = train_or_test
        self.WSI_ids = WSI_ids
        self.scanners = scanners
        
        # dictionary to store all the patches_loader objects
        self.datasets = []
        for scanner in scanners:
            for WSI_id in WSI_ids: 
                ds = patches_loader(train_or_test, WSI_id, scanner, to_torch=True)
                self.datasets.append(ds)

        # index mapping
        self.index_map = []
        for ds_idx, ds in enumerate(self.datasets):
            for i in range(len(ds)):
                self.index_map.append((ds_idx, i))
            
    # define indexing so that the dataloader can access data    
    def __getitem__(self, idx):
        ds_idx, sample_idx = self.index_map[idx]
        return self.datasets[ds_idx][sample_idx] #which is a patches_loader object
    
    def __len__(self):
        return len(self.index_map)

In [4]:
Test = True

if Test: 
    train_scanners = ['Akoya', 'Leica']
    WSI_ids = [1,2]
    target_scanner = ['Akoya'] if 'Akoya' in train_scanners else random.choice(train_scanners)
    train_scanners.remove(target_scanner[0])
    source_scanner = train_scanners
    
    print(target_scanner)
    print(source_scanner)
    
    target_dataset = multi_WSI_loader(WSI_ids, target_scanner, train_or_test='Train')
    source_dataset = multi_WSI_loader(WSI_ids, source_scanner, train_or_test='Train')
    
    
    # DataLoaders
    batch_size = 16
    train_loader_source = DataLoader(source_dataset, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
    train_loader_target = DataLoader(target_dataset, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
    
    
    for batch in train_loader_source:
        images = batch['img']
        labels = batch['label']
        
        print("Source batch - Images shape:", images.shape, "Labels:", labels)
        break
    
    for batch in train_loader_target:
        images = batch['img'].permute(0,3,2,1)
        labels = batch['label']
        embeddings = batch['embedding']
        
        print("Target batch - Images shape:", images.shape, "Labels:", labels, "Emb:", embeddings)
        break
    


['Akoya']
['Leica']
Source batch - Images shape: torch.Size([16, 256, 256, 3]) Labels: tensor([3, 0, 3, 2, 2, 2, 3, 3, 3, 0, 0, 2, 3, 1, 3, 2])
Target batch - Images shape: torch.Size([16, 3, 256, 256]) Labels: tensor([2, 3, 2, 3, 3, 1, 2, 3, 1, 2, 0, 0, 3, 2, 3, 3]) Emb: tensor([[ 0.3118, -0.3488, -0.2644,  ...,  0.5169,  1.7958, -0.2095],
        [ 0.3245, -0.3573, -0.2559,  ...,  0.4809,  1.7637, -0.2082],
        [ 0.3068, -0.3531, -0.2684,  ...,  0.5195,  1.7955, -0.2132],
        ...,
        [ 0.3161, -0.3552, -0.2597,  ...,  0.5087,  1.7853, -0.2083],
        [ 0.3133, -0.3636, -0.2501,  ...,  0.4801,  1.7592, -0.2194],
        [ 0.3209, -0.3543, -0.2612,  ...,  0.5016,  1.7808, -0.2079]])


## Test with network handler


In [5]:
import torch
import torch.nn as nn
import random

class Neural_Network(nn.Module):
    """
    Initialises an Artificial Neural Network with the foundation encoder Gigapath 
    and 2 layers for classification (one to create embeddings, one to classify)

    """

    def __init__(self, BASE_MODEL_DIR, freeze_encoder: bool = True, OT: bool = False, num_classes: int = 5):
        super().__init__() #super constructor for ANN in PyTorch

        self.freeze_encoder = freeze_encoder
        self.OT = OT #maybe not useful here
        
        # Define encoder
        encoder_name = 'gigapath'
        encoder_dir = os.path.join(BASE_MODEL_DIR, "pre_trained_weights")
        encoder_path = os.path.join(encoder_dir, f"{encoder_name}.pth")
        encoder = torch.load(encoder_path, map_location=torch.device("cpu"), weights_only=False)
        self.encoder = encoder

        if self.freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        # Define bottle neck / embeddings
        # fixed parameter value for Gigapath
        in_dim = 1536
        self.bottle_neck = nn.Linear(in_dim, 1024)

        # Define classification head
        out_dim = num_classes
        self.head = nn.Linear(1024, out_dim)

        # Define sequential architecture
        def forward(self, x): 
            if self.freeze_encoder: 
                with torch.no_grad():
                    encoded = self.encoder(x)
                embedding = self.bottle_neck(encoded)
            else:
                embedding = self.bottle_neck(self.encoder(x))
            logits = self.head(embedding)
            return logits


In [6]:
import os
from typing import (
    Tuple, 
    Literal
)
import random
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.utils.data import DataLoader
import deeplake
from sklearn.metrics import balanced_accuracy_score
from geomloss import SamplesLoss

# Ensuring reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False 

# Define global variables
metrics_train = {'running_loss':0, 'predictions':[], 'labels':[]}
metrics_val = {'running_loss':0, 'predictions':[], 'labels':[]}
WSI_ids_train = [1] #to be completed
WSI_ids_val = [2]

# Defining OT-based loss function
loss_geom = SamplesLoss('sinkhorn', p=2, blur=0.1, scaling=0.95, verbose=False)
Lambda = 0.1 # strength of OT (0.1 is the value of the article)

class NetworkHandler:
    '''
    A class to handle training, inference and prediction
    '''

    def __init__(self, precision = 'mixed', freeze_encoder = True, embedding_mode = False, display = False):
        self.precision = precision
        self.freeze_encoder = freeze_encoder
        self.embedding_mode = embedding_mode
        self.display = display

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        BASE_MODEL_DIR = '/home/leolr-int/AGGCPerturbations/model_weights'
        self.model = Neural_Network(BASE_MODEL_DIR)
        self.model = self.model.to(self.device)

        self.use_amp = precision == 'mixed' and self.device == 'cuda'
        self.grad_scaler = GradScaler(enabled=self.use_amp)

    
    def training_no_OT(self, scanners_train):
        # here we train only using cross entropy


        # 1st part: training for one epoch

        optimizer = torch.optim.SGD(self.model.parameters(), lr=0.03, momentum=0.9, weight_decay=0.001)
        torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)

        #data loading
        batch_size = 64 #Careful: the number of data loaded is batch_size * number of scanners 
        dataset_train = multi_WSI_loader(WSI_ids_train, scanners_train, train_or_test='Train')
        loader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)

        
        self.model.train()

        #deactivate the encoder training if needed
        if self.freeze_encoder or self.embedding_mode: 
            self.model.encoder.eval()
        
        pbar = tqdm(loader_train, desc='Training with Cross-Entropy in progress')

        for batch in pbar:
            patch = batch['embedding'] if self.embedding_mode else batch['img'].permute(0,3,2,1) #correct axes for Gigapath
            patch = patch.to(self.device)
            label = batch['label'].to(self.device)

            with torch.autocast(device_type = self.device, dtype = torch.float16, enabled = self.use_amp):
                #if embeddings from gigapath are already computed, we can speed up training
                logits = self.model.bottle_neck(patch) if self.embedding_mode else self.model(patch)
                loss = nn.CrossEntropyLoss()(logits, label) #careful about syntax
            
            self.grad_scaler.scale(loss).backward()
            self.grad_scaler.step(optimizer)
            self.grad_scaler.update()
            optimizer.zero_grad()

            confidence = F.softmax(logits, dim=1)
            pred = torch.argmax(confidence, dim=1)
            
            #performance metrics
            metrics_train['running_loss'] += loss.detach().cpu().item()
            metrics_train['predictions'].extend(pred.cpu().numpy())
            metrics_train['labels'].extend(label.cpu().numpy())

            pbar.set_postfix({'step_loss': loss.detach().cpu().item()})
        
        epoch_loss_train = metrics_train['running_loss'] / len(loader_train)
        epoch_balanced_accuracy_train = balanced_accuracy_score(metrics_train['labels'], metrics_train['predictions'])
        

        # 2nd part: validation for one epoch (WHAT ABOUT torch.no_grad() ??????)
        
        self.model.eval()
        
        #we still work with the Train folder
        dataset_val = multi_WSI_loader(WSI_ids_val, train_scanners, train_or_test='Train')
        loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
        
        pbar = tqdm(loader_val, desc='Validation with Cross-Entropy in progress')
        for batch in pbar:
            patch = batch['embedding'] if self.embedding_mode else batch['img'].permute(0,3,2,1) #correct axes for Gigapath
            patch = patch.to(self.device)
            label = batch['label'].to(self.device)

            with torch.autocast(device_type=self.device, dtype=torch.float16, enabled=self.use_amp):
                logits = self.model.bottle_neck(patch) if self.embedding_mode else self.model(patch)
                loss = nn.CrossEntropyLoss()(logits, label)
            
            confidence = F.softmax(logits, dim=1)
            pred = torch.argmax(confidence, dim=1)

            metrics_val["running_loss"] += loss.detach().cpu().item()
            metrics_val["predictions"].extend(pred.cpu().numpy())
            metrics_val["labels"].extend(label.cpu().numpy())

            pbar.set_postfix({"step_loss": loss.detach().cpu().item()})

        epoch_loss_val = metrics_val["running_loss"] / len(loader_val)
        epoch_balanced_accuracy_val = balanced_accuracy_score(metrics_val["labels"], metrics_val["predictions"])

        
        return epoch_loss_train, epoch_balanced_accuracy_train, epoch_loss_val, epoch_balanced_accuracy_val 
                    
                


 


In [7]:
torch.cuda.empty_cache()
scanners_train = ['Akoya', 'Leica']
train_or_test = 'Train'
WSI_ids = [1] 
batch_size = 64
NetworkHandler(embedding_mode=True).training_no_OT(scanners_train)


'''
trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(trainable_params, lr=args["learning_rate"], weight_decay=args["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, args["epochs"], eta_min=args["eta_min"])
'''

/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Training with Cross-Entropy in progress: 100%|█| 316/316 [02:24<00:00,  2.1
/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
Validation with Cross-Entropy in progress: 100%|█| 164/164 [01:12<00:00,  2


'\ntrainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))\n\ncriterion = nn.CrossEntropyLoss()\noptimizer = torch.optim.AdamW(trainable_params, lr=args["learning_rate"], weight_decay=args["weight_decay"])\nscheduler = CosineAnnealingLR(optimizer, args["epochs"], eta_min=args["eta_min"])\n'

In [8]:
'''min_val_loss, max_val_accuracy = inf, -inf

for epoch in range(1, args["epochs"] + 1):
    print("\n" + f"Epoch [{epoch}/{args['epochs']}]".center(BORDER_WIDTH))
    print(f"{'=' * BORDER_WIDTH}\n")

    writer.add_scalar("Learning Rate", scheduler.optimizer.param_groups[0]["lr"], epoch)

    train_loss, train_accuracy = network_handler.train_epoch(train_loader)
    log_metrics(writer=writer, loss=train_loss, prefix="Train", epoch=epoch, performance=train_accuracy)

    print(f"{'-' * BORDER_WIDTH}\n")

    val_loss, val_accuracy = network_handler.validate_epoch(val_loader)
    log_metrics(writer=writer, loss=val_loss, prefix="Validation", epoch=epoch, performance=val_accuracy)

    if val_loss < min_val_loss:
        torch.save(model.fc.state_dict(), os.path.join(model_dir, f"lowest_loss.pth"))
        min_val_loss = val_loss
        print("New minimum loss — model saved.")
    
    if val_accuracy > max_val_accuracy:
        torch.save(model.fc.state_dict(), os.path.join(model_dir, f"highest_balanced_accuracy.pth"))
        max_val_accuracy = val_accuracy
        print("New maximum balanced accuracy — model saved.")

    scheduler.step()

    print(f"{'=' * BORDER_WIDTH}\n")

print("Run Summary:")
print(f"Min Loss: {min_val_loss:.4f} | Max Balanced Accuracy: {max_val_accuracy:.4f}\n")'''

'min_val_loss, max_val_accuracy = inf, -inf\n\nfor epoch in range(1, args["epochs"] + 1):\n    print("\n" + f"Epoch [{epoch}/{args[\'epochs\']}]".center(BORDER_WIDTH))\n    print(f"{\'=\' * BORDER_WIDTH}\n")\n\n    writer.add_scalar("Learning Rate", scheduler.optimizer.param_groups[0]["lr"], epoch)\n\n    train_loss, train_accuracy = network_handler.train_epoch(train_loader)\n    log_metrics(writer=writer, loss=train_loss, prefix="Train", epoch=epoch, performance=train_accuracy)\n\n    print(f"{\'-\' * BORDER_WIDTH}\n")\n\n    val_loss, val_accuracy = network_handler.validate_epoch(val_loader)\n    log_metrics(writer=writer, loss=val_loss, prefix="Validation", epoch=epoch, performance=val_accuracy)\n\n    if val_loss < min_val_loss:\n        torch.save(model.fc.state_dict(), os.path.join(model_dir, f"lowest_loss.pth"))\n        min_val_loss = val_loss\n        print("New minimum loss — model saved.")\n\n    if val_accuracy > max_val_accuracy:\n        torch.save(model.fc.state_dict(), 